# 多数据集通用数据分析 Agent（DataAnalysisAgent）

> 基于 HelloAgents 框架的智能数据分析助手：输入一个数据文件，自动完成 **加载 → 概览 → 清洗 → 统计 → 分组 → 可视化 → 建模 → 报告** 的完整分析闭环。

## 📝 项目简介

本项目构建了一个**多数据集通用数据分析智能体**：用户只需提供 CSV / Excel 数据文件和一句自然语言需求，Agent 会自主规划分析步骤、调用数据分析工具逐步执行，最终生成带图表的 Markdown 分析报告。

**解决什么问题**：传统数据分析需要人工编写大量 pandas / sklearn 代码，门槛高、流程长。本系统把数据分析流程封装为 8 个可被 LLM 调用的工具，结合 **Plan-and-Solve 智能体范式**，让大模型（deepseek-flash）自主完成规划与执行。

**特色功能**：
- ✅ 8 个数据分析工具：加载 / 概览 / 清洗 / 统计 / 分组 / 可视化 / 建模 / 报告
- ✅ PlanSolveAgent 自主规划：LLM 先生成 3~8 步分析计划，再逐步执行
- ✅ 防目标泄漏：建模时自动剔除标识符与二值代理列
- ✅ 全中文输出：统计结论、图表、报告均为中文

**适用场景**：学习数据分析的自动化演示、通用数据集的快速探索、教学演示。

## 👤 作者信息
- GitHub: [@iamfine78](https://github.com/iamfine78)
- 项目：基于 HelloAgents 的毕业设计


## 环境配置

### 环境要求
- Python 3.10+（本设计在 Python 3.13 验证）
- 一个可用的 LLM API（本项目使用 deepseek-flash）

### 安装依赖


In [1]:
!pip install -q hello_agents==1.0.1 pandas numpy matplotlib openpyxl python-dotenv scikit-learn

ERROR: Ignored the following versions that require a different python version: 1.0.1 Requires-Python >=3.12,<3.14
ERROR: Could not find a version that satisfies the requirement hello_agents==1.0.1 (from versions: 0.1.0, 0.1.1, 0.2.0, 0.2.1, 0.2.2, 0.2.3, 0.2.4, 0.2.5, 0.2.6, 0.2.7, 0.2.8, 0.2.9, 1.0.0)
ERROR: No matching distribution found for hello_agents==1.0.1


In [2]:
import os
import matplotlib
matplotlib.use("Agg")  # 无界面后端，避免 GUI 线程警告
import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv

# 加载 .env 中的 LLM 配置（LLM_MODEL_ID / LLM_API_KEY / LLM_BASE_URL）
load_dotenv()
print("模型:", os.getenv("LLM_MODEL_ID", "(未配置，请在 .env 中设置)"))

模型: deepseek-flash


## 工具定义

定义 8 个数据分析工具，全部封装在一个 `DataAnalysisTool` 类中，通过 `@tool_action` 装饰器自动展开为独立子工具，由 LLM 按需调用：

| 工具 | 作用 |
|---|---|
| `load_data` | 加载 CSV/XLSX，存入会话状态 |
| `overview` | 数据概况：形状 / 类型 / 缺失值 / 前 3 行 |
| `clean_data` | 去重、缺失值填充（数值中位数 / 文本"未知"）、类型转换 |
| `statistics` | 描述统计 + 相关性 Top 对（|r|≥0.3） |
| `group_by` | 分组聚合（mean/sum/count） |
| `visualize` | matplotlib 画图（line/bar/pie）存 `outputs/` |
| `predict` | sklearn 建模（回归 / 分类），防目标泄漏 |
| `write_report` | 汇总所有步骤结果，生成 Markdown 报告 |


In [3]:
from hello_agents.tools import Tool,tool_action



class DataAnalysisTool(Tool):

    def __init__(self):
        super().__init__("data_analysis", "数据分析工具集", expandable=True)
        self.data = None
        self.report_sections = []

    @tool_action("load_data", "加载 CSV/XLSX 数据文件")
    def _load_data(self, file_path: str) -> str:
        '''加载数据文件并保存为会话 DataFrame

        Args:
            file_path: 数据文件路径，支持 CSV 或 Excel(.xlsx/.xls)
        '''
        import pandas as pd

        if file_path.endswith('.csv'):
            self.data = pd.read_csv(file_path)
        elif file_path.endswith(('.xlsx', '.xls')):
            self.data = pd.read_excel(file_path)
        else:
            raise ValueError("Unsupported file format")
        result_text= f"加载成功：{self.data.shape[0]} 行 {self.data.shape[1]} 列\n列名：{', '.join(map(str, self.data.columns))}"
        self.report_sections.append(f"## 数据加载\n\n文件：{file_path}\n{result_text}\n\n")

        return result_text

    @tool_action("overview", "查看数据概况：形状/列类型/缺失值/前几行")
    def overview(self)-> str:
        '''从类变量取 DataFrame → 返回形状 / 列类型 / 缺失值 / 前 3 行摘要'''
        if self.data is None:
            return "尚未加载数据，请先调用 load_data"
        result_text = f"形状：{self.data.shape}\n列类型：{dict(self.data.dtypes)}\n缺失值：{dict(self.data.isnull().sum())}\n前3行：{self.data.head(3).to_dict(orient='records')}"
        self.report_sections.append(f"## 数据概况\n\n{result_text}\n\n")
        return result_text

    def run(self, parameters):
        raise NotImplementedError("可展开工具请调用子工具")

    def get_parameters(self):
        return []

    @tool_action("clean_data", "清理数据：去重、处理缺失（数值填中位数，文本填未知）、类型转换等,返回清洗前后行数对比")
    def clean_data(self)-> str:
        '''清理数据：去重、处理缺失（数值填中位数，文本填未知）、类型转换等'''
        if self.data is None:
            return "尚未加载数据，请先调用 load_data"
        import pandas as pd
        # 去重
        shape_before = self.data.shape
        self.data.drop_duplicates(inplace=True)
        # 处理缺失值（pandas 3.0 CoW：必须用赋值式，不能用链式 inplace）
        for col in self.data.columns:
            if pd.api.types.is_numeric_dtype(self.data[col]):
                median = self.data[col].median()
                self.data[col] = self.data[col].fillna(median)
            else:
                if isinstance(self.data[col].dtype, pd.CategoricalDtype):
                    self.data[col] = self.data[col].cat.add_categories("未知").fillna("未知")
                else:
                    self.data[col] = self.data[col].fillna("未知")
        # 类型转换（尝试将对象列转换为类别型）
        for col in self.data.select_dtypes(include=['object']).columns:
            try:
                self.data[col] = self.data[col].astype('category')
            except Exception as e:
                print(f"转换列 {col} 时出错：{e}")
        result_text = f"清理完成：清理前 {shape_before[0]} 行，清理后 {self.data.shape[0]} 行，当前数据形状 {self.data.shape}"
        self.report_sections.append(f"## 数据清理\n\n{result_text}\n\n")
        return result_text



    @tool_action("statistics", "数据列describe统计+相关系数矩阵，返回文本摘要（相关性 Top 对）")
    def statistics(self)-> str:
        '''返回数据列的 describe 统计信息和相关系数矩阵'''
        if self.data is None:
            return "尚未加载数据，请先调用 load_data"
        # 1. 数值列的描述统计（只挑关键行，控制长度）
        desc = self.data.describe().round(2)
        text = [f"描述统计：{desc.to_string()}"]
        # 2. 相关性只保留 |r| > 0.3 的对
        corr = self.data.select_dtypes(include='number').corr()
        pairs = []
        cols = list(corr.columns)
        for i in range(len(cols)):
            for j in range(i + 1, len(cols)):
                v = corr.iloc[i, j]
                if abs(v) >= 0.3:
                    pairs.append(f"{cols[i]} ↔ {cols[j]}: {v:.2f}")
        text.append("【相关性显著的对】")
        text.extend(pairs if pairs else ["（无明显强相关）"])
        self.report_sections.append(f"## 相关性分析\n\n{'\n'.join(text)}\n\n")
        return "\n".join(text)



    @tool_action("group_by", "分组聚合(mean/sum/count)，返回分组结果文本")
    def group_by(self, group_cols: list, agg_col: str, agg_func: str)->str:
        '''按指定列分组并聚合

        Args:
            group_cols: 分组列列表
            agg_col: 聚合列
            agg_func: 聚合函数，支持 mean/sum/count
        '''
        if self.data is None:
            return "尚未加载数据，请先调用 load_data"
        if not all(col in self.data.columns for col in group_cols + [agg_col]):
            return "指定的列不存在于数据中"
        if agg_func not in ['mean', 'sum', 'count']:
            return "不支持的聚合函数，请使用 mean/sum/count"


        grouped = self.data.groupby(group_cols).agg({agg_col: agg_func}).reset_index()
        result_text = f"分组聚合结果：\n{grouped.to_string(index=False)}"
        self.report_sections.append(f"## 分组聚合\n\n{result_text}\n\n")
        return result_text

    @tool_action("visualize", "matplotlib 画图存 `outputs/` 目录,返回图表文件路径")
    def visualize(self, x_col: str, y_col: str, chart_type: str = 'line')-> str:
        '''可视化数据，支持 line/bar/pie（自动加中文标题、轴标签与图例）

        Args:
            x_col: x 轴列名
            y_col: y 轴列名
            chart_type: 图表类型，支持 line/bar/pie
        '''
        if self.data is None:
            return "尚未加载数据，请先调用 load_data"
        if x_col not in self.data.columns or y_col not in self.data.columns:
            return "指定的列不存在于数据中"
        if chart_type not in ['line', 'bar', 'pie']:
            return "不支持的图表类型，请使用 line/bar/pie"

        import matplotlib.pyplot as plt
        import os

        # 常见列名 → 中文（未知列名回退原英文名）
        COLUMN_ZH = {
            "age": "年龄", "country": "国家", "prior_programming_experience": "先备编程经验",
            "weeks_in_course": "课程周数", "hours_spent_learning_per_week": "每周学习小时数",
            "practice_problems_solved": "已解决练习题数", "projects_completed": "完成项目数",
            "tutorial_videos_watched": "观看教程视频数", "uses_kaggle": "使用Kaggle",
            "participates_in_discussion_forums": "参与讨论论坛", "debugging_sessions_per_week": "每周调试次数",
            "self_reported_confidence_python": "Python信心度", "final_exam_score": "期末成绩",
            "passed_exam": "是否通过考试", "student_id": "学生ID",
        }
        def zh(col: str) -> str:
            return COLUMN_ZH.get(col, col)

        # 确保 outputs 目录存在
        os.makedirs("outputs", exist_ok=True)
        plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']
        plt.rcParams['axes.unicode_minus'] = False

        if chart_type == 'line':
            import pandas as _pd
            import numpy as np
            # 数值化并丢弃缺失，避免字符串/NaN 干扰
            s = _pd.to_numeric(self.data[x_col], errors='coerce')
            y = _pd.to_numeric(self.data[y_col], errors='coerce')
            df = _pd.DataFrame({x_col: s, y_col: y}).dropna()
            if df.empty:
                return "x/y 列无可用的数值数据"
            if df[x_col].nunique() > 20:
                # 连续数值 → 分 10 箱取均值，得到清晰趋势线（避免 500 个点来回折返）
                bins = np.linspace(df[x_col].min(), df[x_col].max(), 11)
                mid = [(bins[i] + bins[i + 1]) / 2 for i in range(10)]
                df['bin'] = _pd.cut(df[x_col], bins=bins, labels=mid, include_lowest=True)
                agg = df.groupby('bin', observed=True)[y_col].mean()
                plt.plot(agg.index.astype(float), agg.values, marker='o', markersize=4)
                plt.xticks(rotation=30)
            else:
                # 离散数值（如 0/1）→ 按值排序取均值
                agg = df.groupby(x_col)[y_col].mean().sort_index()
                plt.plot(agg.index.astype(float), agg.values, marker='o', markersize=4)
            plt.title(f"{zh(y_col)} 随 {zh(x_col)} 变化")
            plt.xlabel(zh(x_col))
            plt.ylabel(zh(y_col))
            plt.grid(True, alpha=0.3)
        elif chart_type == 'bar':
            x_vals = self.data[x_col].fillna("未知").astype(str)  # NaN 先填充再转 str（astype 对 NaN 保留 float）
            plt.bar(x_vals, self.data[y_col])
            plt.title(f"{zh(x_col)} 与 {zh(y_col)} 对比")
            plt.xlabel(zh(x_col))
            plt.ylabel(zh(y_col))
            plt.xticks(rotation=30)
        elif chart_type == 'pie':
            labels = self.data[x_col].fillna("未知").astype(str)
            sizes = self.data[y_col].fillna(0)  # NaN 兜底为 0，避免 Wedge 报错
            plt.pie(sizes, labels=labels, autopct='%1.1f%%')
            plt.title(f"{zh(y_col)} 分布")
        else:
            return "不支持的图表类型，请使用 line/bar/pie"

        # 保存图表
        file_path = f"outputs/{x_col}_{y_col}_{chart_type}.png"
        plt.savefig(file_path, bbox_inches='tight')
        plt.close()
        self.report_sections.append(f"## 可视化\n\n图表：{file_path}\n\n")

        return file_path

    @tool_action("predict", "sklearn 建模 + 评估（R²/ 准确率）,返回模型性能指标")
    def predict(self, target_col: str, task: str)-> str:
        '''使用 sklearn 建模并评估模型性能

        Args:
            target_col: 目标列名
            task: 任务类型，支持 regression/classification
        '''
        if self.data is None:
            return "尚未加载数据，请先调用 load_data"
        if target_col not in self.data.columns:
            return "指定的目标列不存在于数据中"
        if task not in ['regression', 'classification']:
            return "不支持的任务类型，请使用 regression/classification"

        from sklearn.model_selection import train_test_split
        from sklearn.linear_model import LinearRegression, LogisticRegression
        from sklearn.metrics import r2_score, accuracy_score
        import pandas as pd

        # 准备数据（防目标泄漏：只剔除"实际存在"的标识符/代理列，适配任意数据集）
        exclude = {target_col}
        if "student_id" in self.data.columns:
            exclude.add("student_id")
        # 回归时再排除二值代理列（若存在）
        if task == "regression" and "passed_exam" in self.data.columns:
            exclude.add("passed_exam")
        X = self.data.drop(columns=list(exclude))

        y = self.data[target_col]

        # 简单处理非数值列（仅示例，实际应更复杂）
        X = pd.get_dummies(X, drop_first=True)

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        if task == 'regression':
            model = LinearRegression()
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            score = r2_score(y_test, y_pred)
            self.report_sections.append(f"## 模型预测\n\n模型：LinearRegression, R²: {score:.4f}\n\n")
            return f"模型：LinearRegression, R²: {score:.4f}"
        else:  # classification
            model = LogisticRegression(max_iter=1000)
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            score = accuracy_score(y_test, y_pred)
            self.report_sections.append(f"## 模型预测\n\n模型：LogisticRegression, 准确率: {score:.4f}\n\n")
            return f"模型：LogisticRegression, 准确率: {score:.4f}"


    @tool_action("write_report",  "把前面所有步骤的文本汇总成 Markdown 存 outputs/report.md,返回报告文件路径")
    def write_report(self) -> str:
        '''将前面所有步骤的文本汇总成 Markdown 并保存到 outputs/report.md，返回报告文件路径'''
        # 确保 outputs 目录存在
        import os
        os.makedirs("outputs", exist_ok=True)
        report_path = "outputs/report.md"
        with open(report_path, "w", encoding="utf-8") as f:
            if self.data is not None:
                f.write(f"# 数据分析报告\n\n")
                f.write(f"\n\n".join(self.report_sections))
            else:
                f.write("尚未加载数据。\n")
        return report_path


In [4]:
from hello_agents.tools import ToolRegistry

# 注册工具集（自动展开为 8 个独立子工具）
registry = ToolRegistry()
registry.register_tool(DataAnalysisTool())
print("可用工具:", registry.list_tools())

✅ 工具 'data_analysis' 已展开为 8 个独立工具
可用工具: ['load_data', 'clean_data', 'group_by', 'overview', 'predict', 'statistics', 'visualize', 'write_report']


## 智能体构建

采用 **PlanSolveAgent（Plan-and-Solve 范式）**：包含 Planner 与 Executor 两个角色。

- **Planner**：根据用户需求生成 3~8 步分析计划（不写死文件名，按需裁剪标准流程）
- **Executor**：按计划逐步调用 8 个数据分析工具，每步基于工具真实返回继续决策

三层提示词分工：`system_prompt` 定义 Agent 人设与底线，`planner_prompt` 定义分析流程模板，`executor_prompt` 定义工具用法与执行规则。


In [5]:
from hello_agents.core.llm import HelloAgentsLLM
from hello_agents.agents import PlanSolveAgent
from hello_agents.core.config import Config

# 1) 初始化 LLM（自动读取 .env：LLM_MODEL_ID / LLM_API_KEY / LLM_BASE_URL）
llm = HelloAgentsLLM()

# 2) 关闭框架内置工具，只保留 8 个数据分析工具
config = Config(trace_enabled=False, session_enabled=False, skills_enabled=False,
                subagent_enabled=False, todowrite_enabled=False, devlog_enabled=False)

# 3) 三层提示词
system_prompt = """你是一个多数据集通用数据分析助手。你可以分析 CSV 或 Excel 数据文件，
完成数据加载、概览、清洗、统计、分组、可视化、预测建模到生成报告的完整分析流程。
工作方式：
1. 严格依赖工具完成任务，绝不凭记忆编造数据结论；
2. 所有结论必须来自工具返回的真实结果；
3. 输出时用清晰的中文组织，突出关键发现。"""

planner_prompt = """你是一个数据分析规划专家。你的任务是把用户的数据分析需求
分解成多个简单、可执行的步骤，按逻辑顺序排列。

标准分析流程（按需裁剪，不必全部执行）：
1. 数据加载：加载用户指定的数据文件（load_data）；
2. 数据概览：查看规模、列类型、缺失值（overview）；
3. 数据清洗：处理缺失和重复（clean_data，数据不干净或要建模时执行）；
4. 统计分析：描述性统计与相关性（statistics）；
5. 分组分析：按类别列分组对比（group_by，存在有意义类别列时执行）；
6. 可视化：生成关键图表（visualize，选择有分析价值的列组合）；
7. 预测建模：训练模型并评估（predict，用户要求预测或存在目标列时执行）；
8. 报告生成：汇总全部结果（write_report）。

要求：
- 步骤数控制在 3~8 个，每步独立且顺序合理；
- 不写死文件名，从用户需求中提取；
- 根据具体数据和分析目标灵活裁剪。"""

executor_prompt = """你是一个数据分析执行专家。严格按照计划，一步步调用工具完成任务。

可用工具：
- load_data: 加载数据（file_path，支持 CSV/XLSX）。任何分析前必须先加载！
- overview: 数据概况（形状、列类型、缺失值、前几行）
- clean_data: 清洗（去重、缺失值、类型转换）
- statistics: 描述统计与相关性
- group_by: 分组聚合（group_cols 分组列、agg_col 聚合列、agg_func 为 mean/sum/count）
- visualize: 图表存 outputs/（x_col、y_col、chart_type 为 line/bar/pie）
- predict: 预测建模（target_col 目标列、task 为 regression/classification）
- write_report: 生成 Markdown 报告到 outputs/report.md

执行规则：
1. 必须先 load_data，其他工具才能使用；
2. 每次调用后，根据返回的列名和数据类型决定下一步；
3. 专注当前步骤，输出该步骤结果；
4. 最后调用 write_report 生成完整报告。"""

# 4) 构建 PlanSolveAgent
agent = PlanSolveAgent(
    "data_analysis_agent", llm,
    system_prompt=system_prompt,
    planner_prompt=planner_prompt,
    executor_prompt=executor_prompt,
    tool_registry=registry,
    config=config,
    max_tool_iterations=8,
)
print("✅ PlanSolveAgent 构建完成")

✅ PlanSolveAgent 构建完成


## 功能演示

以 Python 学习考试成绩数据集为例（500 名学生 × 15 列），让 Agent 自主完成一次完整的数据分析：

> 任务：分析 `data/python_learning_exam_performance.csv`，找出影响期末成绩的关键因素，并给出可视化图表和完整报告

运行下方单元格约需 1~3 分钟（LLM 规划 + 逐步执行 8 步），期间可观察 Agent 的自主规划与工具调用过程。


In [6]:
# 注意：Jupyter 内核自带事件循环，须用异步入口 arun（脚本环境用 run 即可）
result = await agent.arun(
    "分析 data/python_learning_exam_performance.csv，找出影响期末成绩的关键因素，"
    "并给出可视化图表和完整报告"
)
print(result)

C:\Users\19674\AppData\Local\Temp\ipykernel_31276\864387642.py:67: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in self.data.select_dtypes(include=['object']).columns:


## 步骤 8 结果：报告生成 ✅

报告已保存至 **`outputs/report.md`**。

---

# 📊 分析总结：影响期末成绩的关键因素

## 一、数据概况
- 原始数据 **500 行 × 15 列**，清洗后 **495 行 × 15 列**
- 目标变量：`final_exam_score`（期末成绩），均值 **42.67**，标准差 17.82，通过率仅 **18%**
- 主要数据质量问题：`prior_programming_experience` 缺失率高达 38.2%，且「缺失」本身与低分强相关

## 二、关键发现（按影响力排序）

| 排名 | 因素 | 证据 | 影响力 |
|---|---|---|---|
| 🥇 1 | **编程基础（prior_programming_experience）** | Advanced 64.95 vs Beginner 42.79，差距 **22 分** | ⭐⭐⭐⭐⭐ |
| 🥈 2 | **每周学习时长（hours_spent_learning_per_week）** | 相关系数 **0.41**（最强连续特征） | ⭐⭐⭐⭐ |
| 🥉 3 | **完成项目数（projects_completed）** | 相关系数 **0.37** | ⭐⭐⭐⭐ |
| 4 | **是否使用 Kaggle（uses_kaggle）** | 使用组 45.07 vs 未使用 41.04（+4 分） | ⭐⭐ |
| 5 | 参与讨论区 | 42.91 vs 42.45，差异仅 0.45 分 | ⭐（几乎无影响） |

## 三、建模结论
- 线性回归 **R² = 0.7288**，即模型解释了期末成绩约 **72.9%** 的变异
- 结论稳健：**「既有编程基础」是起点差距，「学习时长 + 项目实战」是最可干预的提升杠杆**

## 四、可执行建议
1. **重点扶持零基础学生**——Beginner 与 Advanced 相差 22 分，是最大差距来源
2. **保障每周学习时长**——最直接的提升路径
3. **强化项目实战**——项目数与成绩正相关，优于被动看教程
4. **引入 Kaggle/Kaggle 类实战平台**——小幅但正向
5. **不必过度强调讨论区打卡**——数据

In [7]:
# 查看生成的报告与图表
import os
print("=== outputs/ 产物清单 ===")
for f in sorted(os.listdir("outputs")):
    size = os.path.getsize(os.path.join("outputs", f)) / 1024
    print(f"  {f}  ({size:.1f} KB)")

=== outputs/ 产物清单 ===
  hours_spent_learning_per_week_final_exam_score_line.png  (29.9 KB)
  prior_programming_experience_final_exam_score_bar.png  (19.9 KB)
  projects_completed_final_exam_score_line.png  (25.8 KB)
  report.md  (6.8 KB)
  uses_kaggle_final_exam_score_bar.png  (13.3 KB)


## 性能评估

### 1. 建模指标解读

演示中回归模型（LinearRegression）的 R² ≈ 0.73。这个值**不是目标泄漏造成的**，而是该数据集本身为"特征线性组合 + 噪声"的合成数据，多特征联合解释力远高于单特征。验证方法如下。

### 2. 目标泄漏对照实验

机器学习中一个关键的质量检查：**用无关目标做对照**。如果建模流程存在隐性泄漏（如行序、索引、编码副作用），无关目标也会被"预测"出高 R²；反之 R² ≈ 0 说明流程干净。


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# 与 clean_data 工具一致的清洗口径（去重 + 缺失填充），确保建模输入无 NaN
df = pd.read_csv("data/python_learning_exam_performance.csv").drop_duplicates()
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].fillna(df[col].median())
    else:
        df[col] = df[col].fillna("未知")

def r2_for(target):
    features = [c for c in df.columns if c not in {target, "student_id", "passed_exam"}]
    X = pd.get_dummies(df[features], drop_first=True)
    Xtr, Xte, ytr, yte = train_test_split(X, df[target], test_size=0.2, random_state=42)
    return r2_score(yte, LinearRegression().fit(Xtr, ytr).predict(Xte))

print("目标=final_exam_score（真实分析目标）:", round(r2_for("final_exam_score"), 4))
print("目标=age（无关对照，应接近 0）     :", round(r2_for("age"), 4))
print()
print("结论：对照实验 R²≈0 证明建模流程无泄漏；")
print("最终成绩 R²≈0.73 来自数据集自身的线性结构（多特征联合解释力）。")

目标=final_exam_score（真实分析目标）: 0.7288
目标=age（无关对照，应接近 0）     : -0.0167

结论：对照实验 R²≈0 证明建模流程无泄漏；
最终成绩 R²≈0.73 来自数据集自身的线性结构（多特征联合解释力）。


## 总结与展望

### 实现的功能
- ✅ 8 个数据分析工具（加载 / 概览 / 清洗 / 统计 / 分组 / 可视化 / 建模 / 报告），支持 CSV 与 Excel
- ✅ PlanSolveAgent 自主规划：LLM 生成 3~8 步分析计划并逐步执行，无需人工编码
- ✅ 全流程自动产出：Markdown 报告 + matplotlib 图表
- ✅ 防目标泄漏建模：自动剔除标识符（student_id）与二值代理列（passed_exam）
- ✅ 泄漏对照实验：用无关目标验证建模流程的可靠性

### 遇到的挑战与解决
| 挑战 | 解决方案 |
|---|---|
| pandas 3.0 Copy-on-Write 导致 `fillna(inplace=True)` 不生效 | 改用赋值式 `df[col] = df[col].fillna(...)` |
| 类别列填充新值报错 | 先 `add_categories` 注册类别再填充 |
| hello_agents 1.0.1 要求 Python <3.14 | 使用 Python 3.13 + hello_agents 1.0.1 |
| LLM 规划出不存在的数据列 | Planner 提示词要求"按实际数据裁剪步骤"，Executor 按工具返回的列名决策 |

### 未来改进方向
- [ ] 支持更多图表类型（散点图、热力图）与自动化图表选择
- [ ] 接入更多模型（随机森林、XGBoost）与交叉验证
- [ ] 增加数据文件自动识别（列语义推断）
- [ ] 多数据集对比分析与综合报告
- [ ] 交互式前端（Streamlit / Gradio）

## 🙏 致谢
感谢 Datawhale 社区与 Hello-Agents 项目提供的高质量学习资源！
